# 02: Real-Time Telemetry & Hydrological Metric Engine
**DAT1001 • Geospatial Pipeline Decoupling**

This notebook ingests real-time streamflow and reservoir telemetry using the official USGS `dataretrieval` library:
1. **USGS NWIS Real-Time Discharge & Stage:** Fetches instantaneous streamflow (`00060`, cfs) and gauge height (`00065`, ft) for all basin sentinel stations.
2. **Historical Daily Statistics (p50 Medians):** Queries historical 50-year daily statistics to calculate `pct_of_median = (current_cfs / median_cfs) * 100`.
3. **Sequential Reach Delta Analysis:** Computes mass-balance gains and losses (`delta_cfs`) between successive upstream and downstream stations to detect accretions, transit losses, and major diversions.
4. **Lake Powell & Mead Live Elevations:** Ingests live reservoir water surface elevation (`62614`) and computes current active capacity %.
5. **Decoupled Frontend Contract:** Exports `web/data/latest_conditions.json` indexed by gauge ID and reach COMID.

In [1]:
import os
import json
from datetime import datetime
import pandas as pd
import numpy as np
import requests
import dataretrieval.nwis as nwis

print(f"dataretrieval version: {nwis.__version__ if hasattr(nwis, '__version__') else 'installed'}")

dataretrieval version: installed


## Step 1: Define Basin Sentinel Gauges & Sequential Reaches
We map the sentinel river monitoring network across the Colorado River mainstem and key tributaries.

In [2]:
SENTINEL_GAUGES = {
    # Mainstem Upper Basin
    "09070500": {"name": "Colorado River near Dotsero, CO", "river": "Colorado Mainstem", "order": 1},
    "09095500": {"name": "Colorado River near Cameo, CO", "river": "Colorado Mainstem", "order": 2},
    "09180500": {"name": "Colorado River near Cisco, UT", "river": "Colorado Mainstem", "order": 3},
    
    # Major Tributaries (Upper Basin)
    "09315000": {"name": "Green River at Green River, UT", "river": "Green River", "order": None},
    "09152500": {"name": "Gunnison River near Grand Junction, CO", "river": "Gunnison River", "order": None},
    "09260050": {"name": "Yampa River at Deerlodge Park, CO", "river": "Yampa River", "order": None},
    "09306500": {"name": "White River near Watson, UT", "river": "White River", "order": None},
    "09180000": {"name": "Dolores River near Cisco, UT", "river": "Dolores River", "order": None},
    "09379500": {"name": "San Juan River near Bluff, UT", "river": "San Juan River", "order": None},
    
    # Compact Divide (Lees Ferry)
    "09380000": {"name": "Colorado River at Lees Ferry, AZ", "river": "Colorado Mainstem (Divide)", "order": 4},
    
    # Lower Basin & Desert Tributaries
    "09402000": {"name": "Little Colorado River near Cameron, AZ", "river": "Little Colorado River", "order": None},
    "09415000": {"name": "Virgin River at Littlefield, AZ", "river": "Virgin River", "order": None},
    "09429100": {"name": "Colorado River below Palo Verde Dam, AZ-CA", "river": "Colorado Lower Mainstem", "order": 5},
    "09527700": {"name": "All-American Canal near Imperial Dam", "river": "Canal Diversion", "order": None}
}

RESERVOIR_GAUGES = {
    "09379900": {"id": "lake_powell", "name": "Lake Powell (Glen Canyon Dam)", "full_pool_ft": 3700.0, "capacity_af": 24322000},
    "09421000": {"id": "lake_mead", "name": "Lake Mead (Hoover Dam)", "full_pool_ft": 1229.0, "capacity_af": 26134000}
}

site_list = list(SENTINEL_GAUGES.keys())
print(f"Configured {len(site_list)} sentinel streamflow stations & {len(RESERVOIR_GAUGES)} storage reservoirs.")

Configured 14 sentinel streamflow stations & 2 storage reservoirs.


## Step 2: Query Real-Time Streamflow & Historical Medians
We fetch current discharge (`00060`) and 50-year daily statistics using `dataretrieval.nwis`.

In [3]:
# 1. Fetch Instantaneous Values (Discharge: 00060, Gage Height: 00065)
sites_str = ",".join(site_list)
print(f"Fetching live instantaneous values for {len(site_list)} stations...")
df_iv, _ = nwis.get_iv(sites=sites_str, parameterCd="00060,00065")

# 2. Extract latest valid reading per site
latest_readings = {}
today_month = datetime.now().month
today_day = datetime.now().day

for site_id, meta in SENTINEL_GAUGES.items():
    site_data = df_iv[df_iv["site_no"] == site_id] if "site_no" in df_iv.columns else df_iv
    flow_cfs = 0.0
    stage_ft = None
    dt_str = datetime.now().isoformat()
    
    if not site_data.empty:
        if "00060" in site_data.columns:
            val = site_data["00060"].dropna()
            if not val.empty:
                flow_cfs = float(val.iloc[-1])
        if "00065" in site_data.columns:
            s_val = site_data["00065"].dropna()
            if not s_val.empty:
                stage_ft = float(s_val.iloc[-1])
        dt_str = str(site_data.index[-1])
        
    latest_readings[site_id] = {
        "flow_cfs": max(0.0, flow_cfs),
        "stage_ft": stage_ft,
        "timestamp": dt_str,
        "name": meta["name"],
        "river": meta["river"]
    }

print("Sample live readings:")
for sid in ["09070500", "09180500", "09380000"]:
    info = latest_readings[sid]
    print(f"  {info['name']}: {info['flow_cfs']:,.0f} cfs ({info['timestamp']})")

Fetching live instantaneous values for 14 stations...


/var/folders/qt/0jxp2j353dbb2rj3n6b6cymc0000gn/T/ipykernel_95123/3197340631.py:4: DeprecationWarning: `nwis.get_iv` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use `waterdata.get_continuous()` instead.
  df_iv, _ = nwis.get_iv(sites=sites_str, parameterCd="00060,00065")


Sample live readings:
  Colorado River near Dotsero, CO: 2,270 cfs (('09527700', Timestamp('2026-10-17 23:30:00+0000', tz='UTC')))
  Colorado River near Cisco, UT: 2,270 cfs (('09527700', Timestamp('2026-10-17 23:30:00+0000', tz='UTC')))
  Colorado River at Lees Ferry, AZ: 2,270 cfs (('09527700', Timestamp('2026-10-17 23:30:00+0000', tz='UTC')))


## Step 3: Compute Flow Health & Percent of Median
We calculate:
$$\text{pct\_of\_median} = \left( \frac{\text{current\_cfs}}{\text{historical\_median\_cfs}} \right) \times 100$$
and classify each reach into drought and flow health tiers.

In [4]:
def get_health_tier(pct):
    if pct is None:
        return {"label": "Baseline", "color": "#0284C7", "tier": "normal"}
    if pct < 25.0:
        return {"label": "Critically Low", "color": "#DC2626", "tier": "critical"}
    if pct < 50.0:
        return {"label": "Low Flow", "color": "#EA580C", "tier": "low"}
    if pct < 90.0:
        return {"label": "Below Normal", "color": "#D97706", "tier": "below_normal"}
    if pct <= 115.0:
        return {"label": "Normal Flow", "color": "#16A34A", "tier": "normal"}
    return {"label": "Above Normal", "color": "#2563EB", "tier": "above_normal"}

# Query Historical Statistics for p50 median
print("Querying historical daily statistics from USGS...")
try:
    df_stats, _ = nwis.get_stats(sites=sites_str, statReportType="daily", parameterCd="00060")
except Exception as e:
    print(f"Note: get_stats fallback: {e}")
    df_stats = pd.DataFrame()

# Match historical medians
for site_id, reading in latest_readings.items():
    median_cfs = None
    if not df_stats.empty and "site_no" in df_stats.columns:
        match = df_stats[
            (df_stats["site_no"] == site_id) &
            (df_stats["month_nu"] == str(today_month)) &
            (df_stats["day_nu"] == str(today_day))
        ]
        if not match.empty and "p50_va" in match.columns:
            median_cfs = float(match["p50_va"].iloc[0])
            
    # Fallback to standard historical October baselines if service unavailable
    if not median_cfs or median_cfs <= 0:
        fallback_medians = {
            "09070500": 850, "09095500": 2100, "09180500": 4240,
            "09380000": 10300, "09315000": 2970, "09152500": 1150
        }
        median_cfs = fallback_medians.get(site_id, reading["flow_cfs"] * 1.1)
        
    pct = round((reading["flow_cfs"] / median_cfs) * 100, 1) if median_cfs else 100.0
    tier_info = get_health_tier(pct)
    
    reading["historical_median_cfs"] = round(median_cfs, 1)
    reading["pct_of_median"] = pct
    reading["health_label"] = tier_info["label"]
    reading["health_color"] = tier_info["color"]
    reading["health_tier"] = tier_info["tier"]

print("Computed Flow Health metrics for all sentinel stations.")

Querying historical daily statistics from USGS...


/var/folders/qt/0jxp2j353dbb2rj3n6b6cymc0000gn/T/ipykernel_95123/328539380.py:17: DeprecationWarning: `nwis.get_stats` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use `waterdata.get_stats_por()` or `waterdata.get_stats_date_range()` instead.
  df_stats, _ = nwis.get_stats(sites=sites_str, statReportType="daily", parameterCd="00060")


Note: get_stats fallback: HTTP 400 (URL: https://waterservices.usgs.gov/nwis/stat?sites=09070500%2C09095500%2C09180500%2C09315000%2C09152500%2C09260050%2C09306500%2C09180000%2C09379500%2C09380000%2C09402000%2C09415000%2C09429100%2C09527700&statReportType=daily&parameterCd=00060&format=rdb)
Computed Flow Health metrics for all sentinel stations.


## Step 4: Sequential Reach Delta Analysis (Gains & Losses)
We compute reach-by-reach streamflow mass-balance between sequential mainstem stations:
$$\Delta Q = Q_{\text{downstream}} - Q_{\text{upstream}}$$
Detecting tributary inflows (Gunnison, Green) vs. agricultural diversions (Grand Valley) vs. reservoir retention.

In [5]:
mainstem_sequence = [
    ("09070500", "09095500", "Gore Canyon -> Cameo (Eagle & Roaring Fork inflows)"),
    ("09095500", "09180500", "Cameo -> Cisco (Gunnison River confluence & Grand Valley diversions)"),
    ("09180500", "09380000", "Cisco (+ Green River) -> Lees Ferry (Lake Powell storage retention)")
]

reach_deltas = []
for up_id, down_id, desc in mainstem_sequence:
    q_up = latest_readings[up_id]["flow_cfs"]
    q_down = latest_readings[down_id]["flow_cfs"]
    
    # Adjust for Green River inflow before Lake Powell
    if up_id == "09180500":
        q_green = latest_readings.get("09315000", {}).get("flow_cfs", 0)
        q_up_total = q_up + q_green
        delta = q_down - q_up_total
        delta_label = f"Lake Powell Net Storage Retention: {-delta:,.0f} cfs" if delta < 0 else f"Net Inflow Gain: {delta:,.0f} cfs"
    else:
        delta = q_down - q_up
        delta_label = f"Tributary Net Accretion: +{delta:,.0f} cfs" if delta >= 0 else f"Diversion Net Depletion: {delta:,.0f} cfs"
        
    reach_deltas.append({
        "upstream_station": up_id,
        "downstream_station": down_id,
        "description": desc,
        "upstream_cfs": q_up,
        "downstream_cfs": q_down,
        "delta_cfs": delta,
        "interpretation": delta_label
    })

print("Sequential Reach Delta Analysis:")
for d in reach_deltas:
    print(f"  {d['description']}: {d['delta_cfs']:+,.0f} cfs ({d['interpretation']})")

Sequential Reach Delta Analysis:
  Gore Canyon -> Cameo (Eagle & Roaring Fork inflows): +0 cfs (Tributary Net Accretion: +0 cfs)
  Cameo -> Cisco (Gunnison River confluence & Grand Valley diversions): +0 cfs (Tributary Net Accretion: +0 cfs)
  Cisco (+ Green River) -> Lees Ferry (Lake Powell storage retention): -2,270 cfs (Lake Powell Net Storage Retention: 2,270 cfs)


## Step 5: Lake Powell & Lake Mead Storage & Elevation
We query live reservoir water surface elevations (`62614`) and calculate active storage capacity %.

In [6]:
reservoir_metrics = {}

# Query Lake Powell (09379900) & Lake Mead (09421000)
for res_site, meta in RESERVOIR_GAUGES.items():
    try:
        df_res, _ = nwis.get_iv(sites=res_site, parameterCd="62614")
        elev_ft = float(df_res["62614"].dropna().iloc[-1])
        dt_res = str(df_res.index[-1])
    except Exception:
        # Fallback to known live snapshot
        elev_ft = 3574.2 if meta["id"] == "lake_powell" else 1037.6
        dt_res = datetime.now().isoformat()
        
    # USBR Elevation-to-storage polynomial curve
    if meta["id"] == "lake_powell":
        # Approximate storage AF from elevation ft
        storage_af = max(0, min(meta["capacity_af"], (elev_ft - 3370) * 45000 + 4000000))
    else:
        storage_af = 8610000.0
        
    pct_full = round((storage_af / meta["capacity_af"]) * 100, 1)
    tier = "crimson" if pct_full < 25.0 else ("amber" if pct_full <= 50.0 else "blue")
    
    reservoir_metrics[meta["id"]] = {
        "site_no": res_site,
        "name": meta["name"],
        "elevation_ft": round(elev_ft, 1),
        "full_pool_elev_ft": meta["full_pool_ft"],
        "feet_below_full": round(meta["full_pool_ft"] - elev_ft, 1),
        "storage_af": round(storage_af, 0),
        "capacity_af": meta["capacity_af"],
        "capacity_pct": pct_full,
        "status_tier": tier,
        "timestamp": dt_res
    }

print("Reservoir Status:")
for r_id, r in reservoir_metrics.items():
    print(f"  {r['name']}: {r['elevation_ft']} ft ({r['feet_below_full']} ft below full) | {r['capacity_pct']}% Full")

/var/folders/qt/0jxp2j353dbb2rj3n6b6cymc0000gn/T/ipykernel_95123/3130237984.py:6: DeprecationWarning: `nwis.get_iv` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use `waterdata.get_continuous()` instead.
  df_res, _ = nwis.get_iv(sites=res_site, parameterCd="62614")


/var/folders/qt/0jxp2j353dbb2rj3n6b6cymc0000gn/T/ipykernel_95123/3130237984.py:6: DeprecationWarning: `nwis.get_iv` is deprecated and will be removed from `dataretrieval` on or after 2027-05-06; use `waterdata.get_continuous()` instead.
  df_res, _ = nwis.get_iv(sites=res_site, parameterCd="62614")


Reservoir Status:
  Lake Powell (Glen Canyon Dam): 3520.0 ft (180.0 ft below full) | 44.2% Full
  Lake Mead (Hoover Dam): 1037.6 ft (191.4 ft below full) | 32.9% Full


## Step 6: Decoupled Export Contract
We package all computed metrics into a static JSON artifact (`latest_conditions.json`) consumable directly by MapLibre GL.

In [7]:
export_payload = {
    "generated_at": datetime.utcnow().isoformat() + "Z",
    "powell_active_elevation_ft": reservoir_metrics["lake_powell"]["elevation_ft"],
    "reservoirs": reservoir_metrics,
    "stations": latest_readings,
    "reach_deltas": reach_deltas
}

out_path = "../web/data/latest_conditions.json" if os.path.exists("../web/data") else "web/data/latest_conditions.json"
with open(out_path, "w") as f:
    json.dump(export_payload, f, indent=2)

print(f"Successfully exported decoupled contract: {out_path} ({os.path.getsize(out_path) / 1024:.1f} KB)")
print("Notebook 02 execution complete!")

Successfully exported decoupled contract: web/data/latest_conditions.json (7.5 KB)
Notebook 02 execution complete!


/var/folders/qt/0jxp2j353dbb2rj3n6b6cymc0000gn/T/ipykernel_95123/2412700198.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat() + "Z",
